In [ ]:
import torch
import torch.optim as optim
# from torchviz import make_dot
import pandas as pd
from itables import init_notebook_mode, show
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import importlib
from tqdm import tqdm

import aacbr
import gradual_aacbr
import semantics.mlp_semantics as ms
import base_scores.feature_weighted_base_score as fwbs
import new_case_influence.nullify_base_scores as nbs
import new_case_influence.base_score_influence as bsi
import edge_weights.identity_edge_weights as iew
import edge_weights.learned_edge_weights as lew

init_notebook_mode(all_interactive=True)

In [ ]:
def reload_imports():
    importlib.reload(aacbr)
    importlib.reload(gradual_aacbr)
    importlib.reload(ms)
    importlib.reload(fwbs)
    importlib.reload(nbs)
    importlib.reload(bsi)
    importlib.reload(iew)
    importlib.reload(lew)

reload_imports()

## Gradual AACBR

In [ ]:
reload_imports()

torch.manual_seed(42)
nodes = torch.rand((4, 5))
nodes[0] = nodes[0] * 0 # Default case has values set to 0 so base_score is computed as 0.5
A = torch.tensor([
                #     0   1   2   3
                    [ 0,  0,  0,  0], # 0
                    [-1,  0,  0,  0], # 1
                    [-1,  0,  0,  0], # 2
                    [ 0, -1,  0,  0], # 3
                ], dtype=torch.float32)

new_cases = torch.rand((1, 5), dtype=torch.float32)

# model = gradual_aacbr.GradualAACBR(fwbs.FeatureWeightedBaseScore(5), ms.MLPBasedSemantics(max_iters=5, epsilon=0), nbs.NullifyBaseScores())
# model = gradual_aacbr.GradualAACBR(fwbs.FeatureWeightedBaseScore(5), ms.MLPBasedSemantics(max_iters=5, epsilon=0), bsi.BaseScoreInfluence(), iew.IdentityEdgeWeights())
model = gradual_aacbr.GradualAACBR(fwbs.FeatureWeightedBaseScore(5), ms.MLPBasedSemantics(max_iters=5, epsilon=0), bsi.BaseScoreInfluence(), lew.LearnedEdgeWeights(4))
result = model(nodes=nodes, A=A, new_cases=new_cases, new_cases_attacks=torch.tensor([True, False, True, True]))
print(result)
# make_dot(result, params=dict(model.named_parameters())).render("attached", format="png")

## Data Set

In [ ]:
SEED = 42

In [ ]:
# from ucimlrepo import fetch_ucirepo 
  
# # fetch dataset 
# connectionist_bench_sonar_mines_vs_rocks = fetch_ucirepo(id=151) 
  
# # data (as pandas dataframes) 
# X = connectionist_bench_sonar_mines_vs_rocks.data.features 
# y = connectionist_bench_sonar_mines_vs_rocks.data.targets 

data = pd.read_csv('data/connectionist-bench-sonar-mines-vs-rocks/sonar.all-data')

data = data.values

X = np.array(data[:, :-1], dtype=np.float32)
y = np.array(data[:, -1])

show(X)
print(np.unique(y))



In [ ]:
encoder = LabelEncoder()
encoder.fit(y)
y = encoder.transform(y)


## Train Model

### Split into Training, Validation and Test

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=SEED)
print(f"Test Size:  {len(X_test)}")
print(f"Train Size:  {len(X_train)}")
print(f"Validation Size:  {len(X_val)}")

In [ ]:
print(X_train)

### Build AF

In [ ]:
# TODO: Consider more sophisticated orders

# Compare against the average for each column
means = X_train.mean(axis=0)
std = X_train.std(axis=0)

STD_PARAM = 2

def binarise_by_normal(case):
    return np.where(np.abs(case - means) <= STD_PARAM*std, 0, 1)


def strictsuperset(a, b):

    if b.ndim == 1:
        b = np.expand_dims(b, axis = 0)

    anb = a & b
    return np.logical_and(np.all(anb == b, axis = -1), np.logical_not(np.all(anb == a, axis = -1)))

In [ ]:
COMPARISON_FUNC = strictsuperset
PREPROCESS_FUNC = binarise_by_normal 

In [ ]:
def plot_weights(model):
    model.compute_base_scores.plot_parameters()

In [ ]:
def plot_edge_weights(model):
    model.compute_edge_weights.plot_parameters()

### Train Model

In [ ]:
DEFAULT_OUTCOME = 1
DEFAULT_CASE = means.copy()
MAX_ITERS = 20 
EPOCHS = 75 
N_SPLITS = 5
USE_SYMMETRIC_ATTACKS = False

In [ ]:
reload_imports()
def build_aacbr(X_train, y_train, casebase_indices):

    aacbr_model = aacbr.AACBR(
        PREPROCESS_FUNC(X_train), 
        y_train, 
        COMPARISON_FUNC, 
        PREPROCESS_FUNC(DEFAULT_CASE), 
        [DEFAULT_OUTCOME],
        use_symmetric_attacks=USE_SYMMETRIC_ATTACKS,
        build_parallel=True,
        casebase_indices=casebase_indices
    )

    default_indexes = aacbr_model.default_indexes

    casebase_tensor = torch.zeros((X_train.shape[0] + len(default_indexes), X_train.shape[1]))
    casebase_tensor[default_indexes, :] = torch.tensor(DEFAULT_CASE)  
    non_default_indexes = np.setdiff1d(np.arange(X_train.shape[0]), default_indexes)
    casebase_tensor[non_default_indexes, :] = torch.tensor(X_train)
    # TODO: Need to handle default better - maybe should have a base score of
    # near 1 - cannot be exactly 1 as MLP semantics do not update if set to 0 or 1
    A = torch.tensor(aacbr_model.A, dtype=torch.float32)


    return casebase_tensor, A, default_indexes, aacbr_model

def run_gradual_model(gradual_model, aacbr_model, casebase_tensor, A, new_cases):

    new_cases_attacks = torch.tensor(
        aacbr_model.get_new_case_attacks_mask(PREPROCESS_FUNC(new_cases))
    )
    new_cases = torch.tensor(new_cases)
    return gradual_model(casebase_tensor, A, new_cases, new_cases_attacks)

def evaluate_model(X_train, y_train, casebase_indices, new_cases, new_cases_labels, show_confusion=False, print_graph=False, print_matrix=False):
    casebase_tensor, A, default_indexes, aacbr_model = build_aacbr(X_train, y_train, casebase_indices)


    final_stengths = run_gradual_model(model, aacbr_model, casebase_tensor, A, new_cases)
    predicted = final_stengths[:, default_indexes].detach().numpy()
    predicted = np.where(predicted > 0.5, 1, 0)

    print("Accuracy, Precision, Recall, F1")
    print([
        accuracy_score(new_cases_labels, predicted),
        precision_score(new_cases_labels, predicted, average='macro'),
        recall_score(new_cases_labels, predicted, average='macro'),
        f1_score(new_cases_labels, predicted, average='macro')
    ])


    if show_confusion:
        cm = confusion_matrix(new_cases_labels, predicted)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot()
        plt.show()

    if print_graph:
        aacbr_model.show_graph_with_labels()

    if print_matrix:
        aacbr_model.show_matrix()



In [ ]:
reload_imports()
torch.manual_seed(0) # TRY DIFFERENT INITIAL WEIGHTS 
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# model = gradual_aacbr.GradualAACBR(fwbs.FeatureWeightedBaseScore(X_train.shape[1]), ms.MLPBasedSemantics(max_iters=MAX_ITERS, epsilon=0), nbs.NullifyBaseScores(), iew.IdentityEdgeWeights())
# model = gradual_aacbr.GradualAACBR(fwbs.FeatureWeightedBaseScore(X_train.shape[1]), ms.MLPBasedSemantics(max_iters=MAX_ITERS, epsilon=0), bsi.BaseScoreInfluence(), iew.IdentityEdgeWeights())
# model = gradual_aacbr.GradualAACBR(fwbs.FeatureWeightedBaseScore(X_train.shape[1]), ms.MLPBasedSemantics(max_iters=MAX_ITERS, epsilon=0), bsi.BaseScoreInfluence(), lew.LearnedEdgeWeights(X_train.shape[0] + 1))
model = gradual_aacbr.GradualAACBR(fwbs.FeatureWeightedBaseScore(X_train.shape[1]), ms.MLPBasedSemantics(max_iters=MAX_ITERS, epsilon=0), nbs.NullifyBaseScores(), lew.LearnedEdgeWeights(X_train.shape[0] + 1))
criterion = torch.nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

In [ ]:
plot_edge_weights(model)

In [ ]:
# Visulaise Feature weights before training
plot_weights(model)

In [ ]:
reload_imports()
with torch.no_grad():
    evaluate_model(X_train, y_train, np.arange(len(X_train)), X_val, y_val, show_confusion=True, print_matrix=True)

In [ ]:
reload_imports()
model.train()
losses = []
for fold, (casebase_index,  new_cases_index) in enumerate(kf.split(X_train)):

    print("Fold:", fold)

    # casebase = X_train[casebase_index]
    # casebase_labels = y_train[casebase_index]

    new_cases = X_train[new_cases_index]
    new_cases_labels = y_train[new_cases_index]

    casebase_tensor, A, default_indexes, aacbr_model = build_aacbr(X_train, y_train, casebase_index)
    new_cases_labels = torch.tensor(new_cases_labels, dtype=torch.float32)

    pbar = tqdm(range(EPOCHS))

    for epoch in pbar:  

        running_loss = 0.0
        optimizer.zero_grad()

        # TODO: Consider batching new_cases?
        final_strengths = run_gradual_model(model, aacbr_model, casebase_tensor, A, new_cases)
        default_strengths = final_strengths[:, default_indexes].squeeze()

        loss = criterion(default_strengths, new_cases_labels)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        losses.append(running_loss/len(new_cases))

        pbar.set_description(f'Epoch {epoch + 1}, Loss: {round(running_loss/len(new_cases), 4)}')




   

print('Finished Training')

plt.plot(losses)
plt.show()


In [ ]:
plot_edge_weights(model)

In [ ]:
plot_weights(model)

In [ ]:
reload_imports()
with torch.no_grad():
    evaluate_model(X_train, y_train, np.arange(len(X_train)), X_val, y_val, show_confusion=True, print_matrix=True)

### Test Set

In [ ]:
# reload_imports()
# with torch.no_grad():
#     evaluate_model(X_train_full, y_train_full, X_test, y_test, show_confusion=True)